# Wage and salary income in North Carolina

This notebook gets person records from the 2024 ACS
1-Year PUMS API, records each cleaning decision, and creates descriptive
tables and charts.
Source: https://api.census.gov/data/2024/acs/acs1/pums.html
Dictionary: https://www2.census.gov/programs-surveys/acs/tech_docs/pums/data_dict/PUMS_Data_Dictionary_2024.pdf

In [ ]:
from getpass import getpass
from pathlib import Path
from shutil import make_archive
import os

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd
import requests

OUTPUT = Path.cwd() / "portfolio_outputs"
OUTPUT.mkdir(exist_ok=True)
API_URL = "https://api.census.gov/data/2024/acs/acs1/pums"
FIELDS = ["AGEP", "SEX", "WAGP", "ADJINC", "SCHL", "OCCP", "INDP", "WKHP", "WKWN", "PWGTP"]
api_key = os.environ.get("CENSUS_API_KEY") or getpass("Paste your Census API key here (hidden): ")
if not api_key.strip():
    raise ValueError("A Census API key is required. Request one at https://api.census.gov/data/key_signup.html.")

## 1. Collect the data through the Census API

`state:37` selects North Carolina. It also saves a raw CSV

In [ ]:
try:
    response = requests.get(API_URL, params={"get": ",".join(FIELDS), "for": "state:37", "key": api_key}, timeout=180)
    if not response.ok:
        raise ValueError()
    payload = response.json()
    if not isinstance(payload, list) or len(payload) < 2:
        raise ValueError()
except (requests.RequestException, ValueError):
    raise RuntimeError("Census API request did not work. Check your key and connection.") from None
raw = pd.DataFrame(payload[1:], columns=payload[0])
raw.to_csv(OUTPUT / "raw_nc_2024_pums.csv", index=False)
print(f"Census API request worked: {len(raw):,} person records downloaded.")
display(raw.head())

## 2. Inspect missing data and filter one rule at a time

Some Census fields use a blank or `N` for not applicable. Occupation and
industry are kept as stringsThe filter log shows how many records each decision removes.

In [ ]:
numeric_fields = ["AGEP", "SEX", "WAGP", "ADJINC", "SCHL", "WKHP", "WKWN", "PWGTP"]
df = raw.copy()
for field in numeric_fields:
    df[field] = pd.to_numeric(df[field], errors="coerce")
for field in ["OCCP", "INDP"]:
    df[field] = df[field].replace({"": pd.NA, "N": pd.NA})
missing = df[FIELDS].isna().sum().rename_axis("field").reset_index(name="missing_records")
missing.to_csv(OUTPUT / "missing_values.csv", index=False)
display(missing)

# Log each filter separately so the number of excluded records is visible.
audit = [{"step": "Downloaded NC person records", "remaining": len(df), "removed_at_step": 0}]

def keep_rows(label, condition):
    global df
    previous = len(df)
    df = df.loc[condition].copy()
    audit.append({"step": label, "remaining": len(df), "removed_at_step": previous - len(df)})

keep_rows("Age 25 to 64", df["AGEP"].between(25, 64))
keep_rows("Census sex code 1 or 2", df["SEX"].isin([1, 2]))
keep_rows("Education code 01 to 24", df["SCHL"].between(1, 24))
keep_rows("Positive wage and salary income", df["WAGP"].gt(0))
keep_rows("Positive income adjustment factor", df["ADJINC"].gt(0))
keep_rows("Positive person survey weight", df["PWGTP"].gt(0))
audit_table = pd.DataFrame(audit)
audit_table.to_csv(OUTPUT / "filter_audit.csv", index=False)
display(audit_table)
print(f"Final analytic records: {len(df):,}")

## 3. Define the measures

The Census bulk CSV stores `ADJINC` as `1015250` (six implied decimal
places), while this API response stores it as `1.015250`. The code checks
which scale arrived before multiplying `WAGP` by the adjustment factor.
The four education groups follow the 2024 PUMS data dictionary.

In [ ]:
# The API and bulk CSV use different numeric scales for ADJINC.
if df["ADJINC"].between(500_000, 2_000_000).all():
    df["income_adjustment"] = df["ADJINC"] / 1_000_000
elif df["ADJINC"].between(0.5, 2).all():
    df["income_adjustment"] = df["ADJINC"]
else:
    raise ValueError("Unexpected ADJINC scale. Inspect the raw API response before calculating wages.")
df["wage_2024"] = df["WAGP"] * df["income_adjustment"]
print("Income adjustment factor(s):", df["income_adjustment"].drop_duplicates().tolist())
df["sex_label"] = df["SEX"].map({1: "Men", 2: "Women"})
education_order = [
    "High school or less", "Some college or associate degree",
    "Bachelor's degree", "Graduate or professional degree",
]
age_order = ["25-34", "35-44", "45-54", "55-64"]
df["education_group"] = pd.cut(
    df["SCHL"], bins=[0, 17, 20, 21, 24], labels=education_order,
)
df["age_group"] = pd.cut(
    df["AGEP"], bins=[24, 34, 44, 54, 64], labels=age_order,
)
df.to_csv(OUTPUT / "cleaned_nc_2024_pums.csv", index=False)
display(df[["AGEP", "sex_label", "WAGP", "ADJINC", "wage_2024", "education_group", "age_group", "PWGTP"]].head())

## 4. Calculate weighted medians and explicit gaps

Some important definitions: The weighted median is the first wage value where cumulative person weights
reach half the group's total weight. Group counts are unweighted record
counts. A percentage gap is calculated as `(men's median - women's median) / men's median`.

In [ ]:
# The weighted median is the first wage reaching half the total person weight.
def weighted_median(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    order = np.argsort(values, kind="stable")
    values, weights = values[order], weights[order]
    return float(values[np.searchsorted(np.cumsum(weights), weights.sum() / 2, side="left")])

def summarize(data, group_fields):
    rows = []
    for label, part in data.groupby(group_fields, observed=True, sort=False):
        if not isinstance(label, tuple):
            label = (label,)
        row = dict(zip(group_fields, label))
        row.update({
            "unweighted_records": len(part),
            "weighted_median_2024_dollars": weighted_median(part["wage_2024"], part["PWGTP"]),
        })
        rows.append(row)
    return pd.DataFrame(rows)

def gap_table(summary, group_field=None):
    groups = [None] if group_field is None else summary[group_field].drop_duplicates().tolist()
    rows = []
    for group in groups:
        pair = summary if group_field is None else summary.loc[summary[group_field] == group]
        if set(pair["sex_label"]) != {"Men", "Women"}:
            continue
        men = pair.loc[pair["sex_label"] == "Men"].iloc[0]
        women = pair.loc[pair["sex_label"] == "Women"].iloc[0]
        m = men["weighted_median_2024_dollars"]
        w = women["weighted_median_2024_dollars"]
        row = {
            "men_median": m, "women_median": w,
            "dollar_gap": m - w, "gap_percent_of_men_median": 100 * (m - w) / m,
            "men_records": int(men["unweighted_records"]),
            "women_records": int(women["unweighted_records"]),
        }
        if group_field is not None:
            row = {group_field: group, **row}
        rows.append(row)
    return pd.DataFrame(rows)

overall = gap_table(summarize(df, ["sex_label"]))
education = gap_table(summarize(df, ["education_group", "sex_label"]), "education_group")
age = gap_table(summarize(df, ["age_group", "sex_label"]), "age_group")
for name, table in [("overall", overall), ("education", education), ("age", age)]:
    table.to_csv(OUTPUT / f"{name}_summary.csv", index=False)
    print(f"{name.title()} summary")
    display(table.round(2))

## 5. Select well-represented occupations

Occupation codes are four-character strings. The six displayed codes are the
most common among records with at least 100 men and 100 women in the
analytic sample. Their labels were checked against the 2024 PUMS dictionary.
This selection helps avoid tiny groups, but it does not make occupations
comparable in hours, seniority, or job duties.

In [ ]:
occupation_labels = {
    "0440": "Other managers",
    "3255": "Registered nurses",
    "2310": "Elementary/middle school teachers",
    "9130": "Driver/sales workers and truck drivers",
    "4700": "First-line retail sales supervisors",
    "5240": "Customer service representatives",
}
counts = df.groupby(["OCCP", "sex_label"], observed=True).size().unstack(fill_value=0)
# Require at least 100 sampled people in both groups before plotting an occupation.
eligible = counts.loc[counts[["Men", "Women"]].min(axis=1).ge(100)].copy()
selected_codes = eligible.sum(axis=1).nlargest(6).index.tolist()
if set(selected_codes) != set(occupation_labels):
    raise RuntimeError(
        f"Occupation selection changed: {selected_codes}. Check these codes against the Census dictionary before reporting results."
    )
occupation_data = df.loc[df["OCCP"].isin(selected_codes)].copy()
occupation_data["occupation"] = occupation_data["OCCP"].map(occupation_labels)
occupation = gap_table(summarize(occupation_data, ["occupation", "sex_label"]), "occupation")
occupation_order = [occupation_labels[code] for code in selected_codes]
occupation.to_csv(OUTPUT / "occupation_summary.csv", index=False)
print("Occupation summary")
display(occupation.round(2))

## 6. Create four charts

Each chart uses the same summaries as the tables above.

In [ ]:
MEN, WOMEN = "#276b70", "#ac573e"
dollars = FuncFormatter(lambda value, _: f"${value:,.0f}")
plt.rcParams.update({"font.size": 10, "figure.facecolor": "#fffdf9", "axes.facecolor": "#fffdf9", "axes.spines.top": False, "axes.spines.right": False})

fig, ax = plt.subplots(figsize=(8, 4.5))
vals = [overall.iloc[0]["men_median"], overall.iloc[0]["women_median"]]
bars = ax.barh(["Men", "Women"], vals, color=[MEN, WOMEN], height=.55)
ax.invert_yaxis()
ax.bar_label(bars, labels=[f"${v:,.0f}" for v in vals], padding=6)
ax.set_xlim(0, max(vals) * 1.2)
ax.set_xlabel("Median annual wage and salary income (2024 dollars)")
ax.set_title("North Carolina: a 25% difference in annual wage medians", loc="left")
ax.xaxis.set_major_formatter(dollars)
ax.grid(axis="x", color="#ded8cb", alpha=.7)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(OUTPUT / "figure1_overall.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(10, 5.3))
y = np.arange(len(education_order))
e = education.set_index("education_group").loc[education_order]
ax.hlines(y, e["women_median"], e["men_median"], color="#c8beb0", lw=2)
ax.scatter(e["men_median"], y, s=110, color=MEN, label="Men", zorder=3)
ax.scatter(e["women_median"], y, s=95, color=WOMEN, marker="s", label="Women", zorder=3)
for i, row in enumerate(e.itertuples()):
    ax.text(row.men_median + 2500, i, f"{row.gap_percent_of_men_median:.0f}% gap", va="center", fontsize=9)
ax.set_yticks(y, education_order)
ax.invert_yaxis()
ax.set_xlim(0, e["men_median"].max() * 1.3)
ax.set_xlabel("Median annual wage and salary income (2024 dollars)")
ax.set_title("Wage differences persist across education levels", loc="left")
ax.xaxis.set_major_formatter(dollars)
ax.grid(axis="x", color="#ded8cb", alpha=.7)
ax.set_axisbelow(True)
ax.legend(title="Reported sex", frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT / "figure2_education.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
a = age.set_index("age_group").loc[age_order]
ax.plot(age_order, a["men_median"], marker="o", markersize=9, lw=2.5, color=MEN, label="Men")
ax.plot(age_order, a["women_median"], marker="s", markersize=8, lw=2.5, color=WOMEN, label="Women")
ax.set_xlabel("Age group (years)")
ax.set_ylabel("Weighted median annual wage income (2024 dollars)")
ax.set_title("Largest age-group difference occurs at 55–64", loc="left")
ax.yaxis.set_major_formatter(dollars)
ax.grid(axis="y", color="#ded8cb", alpha=.7)
ax.set_axisbelow(True)
ax.legend(title="Reported sex", frameon=False, loc="upper left")
ax.text(.98, .56, f"55–64 difference: ${a.loc['55-64', 'dollar_gap']:,.0f}", transform=ax.transAxes, ha="right", va="center", fontsize=10, bbox={"facecolor": "#fffdf9", "edgecolor": "#ded8cb", "boxstyle": "round,pad=.4"})
fig.tight_layout()
fig.savefig(OUTPUT / "figure3_age.png", dpi=200)
plt.show()

## 7. Compare selected occupations

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
o = occupation.set_index("occupation").loc[occupation_order]
y = np.arange(len(occupation_order))
height = .34
ax.barh(y - height / 2, o["men_median"], height=height, color=MEN, label="Men")
ax.barh(y + height / 2, o["women_median"], height=height, color=WOMEN, label="Women")
for i, row in enumerate(o.itertuples()):
    ax.text(max(row.men_median, row.women_median) + 2000, i, f"{row.gap_percent_of_men_median:.0f}% gap", va="center", fontsize=9)
ax.set_yticks(y, occupation_order)
ax.invert_yaxis()
ax.set_xlim(0, o[["men_median", "women_median"]].max().max() * 1.3)
ax.set_xlabel("Median annual wage and salary income (2024 dollars)")
ax.set_title("Wage differences vary across six selected occupations", loc="left")
ax.xaxis.set_major_formatter(dollars)
ax.grid(axis="x", color="#ded8cb", alpha=.7)
ax.set_axisbelow(True)
ax.legend(title="Reported sex", frameon=False)
fig.tight_layout()
fig.savefig(OUTPUT / "figure4_occupation.png", dpi=200)
plt.show()

## 8. A work-time check

This extra comparison keeps only records reporting at least 35 usual hours
per week and 50 weeks in the past year. It does **not** turn annual income
into hourly pay or control for every job difference. It is a sensitivity
check, not a causal adjustment.

In [ ]:
full_time_year_round = df.loc[df["WKHP"].ge(35) & df["WKWN"].ge(50)].copy()
full_time_result = gap_table(summarize(full_time_year_round, ["sex_label"]))
full_time_result.to_csv(OUTPUT / "full_time_year_round_summary.csv", index=False)
print("Full-time, year-round sensitivity check")
display(full_time_result.round(2))

archive = make_archive("portfolio_outputs_2024", "zip", OUTPUT)
print(f"Charts and tables saved in {OUTPUT}. ZIP: {archive}")